# Classic RAG Workflow


In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import from our application modules
from src.chunking.parent_child import ingest
from src.embedding.embed import upsert_chunks
from src.retrieval.retriever import search_vector_db
from src.generation.generator import generate_answer
from src.config import parent_store_collection
from src.caching.semantic_cache import get_semantic_cache, set_semantic_cache

### 1. Ingestion

In [ ]:
file_to_ingest = Path(r"C:\Users\amanm\Desktop\learning\developer-chat-agent\RAGAnything.pdf") # Replace with your file path
namespace = "default_namespace" 
document_id = "doc_1"

if file_to_ingest.exists():
    print(f"Ingesting {file_to_ingest}...")
    
    records, parents = ingest(file_to_ingest)
    
    print("Storing parent chunks in MongoDB...")
    for pid, text in parents.items():
        parent_store_collection.update_one(
            {"parent_id": pid, "namespace": namespace},
            {"$set": {"text": text, "document_id": document_id}},
            upsert=True
        )
        
    print("Upserting child chunks to Pinecone...")
    upsert_chunks(
        chunks=records, 
        namespace=namespace, 
        document_id=document_id
    )
    print("Ingestion complete!")
else:
    print(f"File {file_to_ingest} not found! Please point it to a valid document.")

### 2. Retrieval & Generation with Semantic Caching

In [ ]:
query = "What is the main topic of the document?"
namespace = "default_namespace"

# Semantic Cache (Tier 2)
cached_answer, cached_sources, query_emb = get_semantic_cache(query=query)

if cached_answer:
    print("--- SEMANTIC CACHE HIT ---")
    print(cached_answer)
else:
    print("--- CACHE MISS. RETRIEVING & GENERATING ---")

    retrieved_chunks = search_vector_db(
        namespace=namespace, 
        query=query, 
        top_k=5
    )

    if retrieved_chunks:
        answer = generate_answer(query, retrieved_chunks, namespace)
        print("\n--- LLM ANSWER ---")
        print(answer)
        
        sources = list(set([chunk.get("source") for chunk in retrieved_chunks]))
        set_semantic_cache(
            query=query, 
            answer=answer, 
            sources=sources,
            query_emb=query_emb
        )
    else:
        print("No relevant context found to answer the query.")